# Ghostexec + Unsloth + GRPO (Colab)

This notebook mirrors the **install stack** from Unsloth’s [OpenEnv 2048 / GRPO Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb): **`uv`**, **Unsloth + Unsloth Zoo from GitHub**, and pinned **`transformers==4.56.2`** + **`trl==0.22.2`** so `GRPOTrainer` imports cleanly (avoids newer TRL + `mergekit` + Pydantic issues on Colab).

**Hardware:** **GPU** runtime (T4 or better). **Runtime → Run all** from the top.

1. **Optional SFT** — synthetic briefing → `GhostexecAction` JSON (`smart_action` from `training/train.py`; upgrade to rejection-sampled SFT via `training.rejection_sft`).
2. **Optional constrained decoding** — `patch_model_for_json_generation` enforces the `GhostexecAction` schema at sampling time. `GHOSTEXEC_CONSTRAIN_JSON=1` + `pip install lm-format-enforcer`.
3. **BEFORE eval** — untrained LLM policy on the held-out scenario (`vip_meltdown.json`) + fixed-briefing action sample.
4. **GRPO** — list of reward channels from `training.grpo_ghostexec_reward`: core `ghostexec_env_step_reward` + `reward_format_valid`, `reward_group_diversity`, `reward_id_relevance`, `reward_vip_critical_reply_bonus`. Knobs: `GHOSTEXEC_CURRICULUM`, `GHOSTEXEC_PERTURB`, `GHOSTEXEC_REWARD_KSTEPS`, `GHOSTEXEC_MULTITURN`.
5. **Plots** — reward curve + per-channel breakdown saved to `outputs/plots/*.png`.
6. **AFTER eval + comparison** — trained LLM policy on the same held-out scenario, `before_after.csv` delta table, and a side-by-side action sample. These are the artifacts for the *observable evidence of training progress* rubric.

**Docs:** [Unsloth RL guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) · **Scope:** first-step comparisons on a deterministic reset (`phase2_core.json` by default).

**Repo:** set `GHOSTEXEC_REPO_URL` (or edit `GITHUB_REPO_URL` in the knobs cell) if you are not already inside the cloned `ghostexec` folder under `/content/ghostexec`.

**Kaggle:** turn **Internet** on in notebook settings to clone from GitHub; or add this repo as a **Dataset** under `/kaggle/input/...` and run the **Notebook bootstrap** cell so `from ghostexec...` works (fixes `ModuleNotFoundError: No module named 'ghostexec'` when `cwd` is `/kaggle/working`).


## Installation (same pins as OpenEnv 2048 notebook)

Uses **`uv pip`** like the reference notebook: fresh Colab gets **torch**, **bitsandbytes**, **transformers 4.56.2**, **Unsloth from git**, then **`trl==0.22.2`** with **`--no-deps`** so pins stick.

Cells use **plain Python** (no `!` / `%%` magics) so the file stays easy to validate offline.


In [ ]:
import os

GITHUB_REPO_URL = os.environ.get("GHOSTEXEC_REPO_URL", "https://github.com/false200/ghostexec.git").strip()

RUN_SFT = os.environ.get("GHOSTEXEC_RUN_SFT", "1") != "0"
SFT_SAMPLES = int(os.environ.get("GHOSTEXEC_SFT_SAMPLES", "256"))
SFT_MAX_STEPS = int(os.environ.get("GHOSTEXEC_SFT_MAX_STEPS", "100"))
GRPO_DATASET_ROWS = int(os.environ.get("GHOSTEXEC_GRPO_ROWS", "128"))
GRPO_MAX_STEPS = int(os.environ.get("GHOSTEXEC_GRPO_MAX_STEPS", "200"))
NUM_GENERATIONS = int(os.environ.get("GHOSTEXEC_NUM_GENERATIONS", "6"))

MODEL_NAME = os.environ.get(
    "GHOSTEXEC_MODEL",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
)
MAX_SEQ_LENGTH = int(os.environ.get("GHOSTEXEC_MAX_SEQ", "2048"))

# Note: ``GHOSTEXEC_GRPO_MAX_STEPS``, ``GHOSTEXEC_GRPO_ROWS``, ``GHOSTEXEC_NUM_GENERATIONS`` and
# ``GHOSTEXEC_GRPO_TEMPERATURE`` / ``GHOSTEXEC_GRPO_MAX_COMPLETION_LENGTH`` are
# re-read from ``os.environ`` again at the start of the GRPO cell (so a later cell can override).


In [ ]:
import shutil
import subprocess

if shutil.which("apt-get"):
    subprocess.run(
        ["apt-get", "update"],
        check=False,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "ffmpeg", "libavcodec-extra"],
        check=False,
    )
else:
    print("apt-get not found; skipping ffmpeg install (OK on Windows / some Kaggle images).")


## Ghostexec repository

Clone into `/content/ghostexec` (Colab) or skip if you already uploaded the repo there. Then `pip install -e .` and **re-apply** the TRL / Transformers pins so dependencies from `pyproject.toml` do not float TRL upward.

On **Kaggle**, if imports fail with `No module named 'ghostexec'`, run the **Notebook bootstrap** cell next (or add the repo as a Dataset under `/kaggle/input`).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

target = Path("/content/ghostexec")
candidates = [Path.cwd(), Path.cwd() / "ghostexec", target]
root = None
for p in candidates:
    if (p / "pyproject.toml").exists() and (p / "models.py").exists():
        root = p.resolve()
        break

if root is None:
    if not GITHUB_REPO_URL or "YOUR_ORG" in GITHUB_REPO_URL:
        raise RuntimeError(
            "Could not find ghostexec. Set GHOSTEXEC_REPO_URL / edit GITHUB_REPO_URL, or upload the repo."
        )
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(target)], check=True)
    root = target.resolve()

os.chdir(root)
sys.path.insert(0, str(root))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(root), check=True)
# Install uv first (Colab does not ship it; required for `python -m uv pip ...`).
subprocess.run([sys.executable, "-m", "pip", "install", "-qqq", "uv"], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "pip",
        "install",
        "--upgrade",
        "--no-deps",
        "transformers==4.56.2",
        "tokenizers",
        "trl==0.22.2",
        "unsloth",
        "unsloth_zoo",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "-qqq", "matplotlib", "datasets"],
    check=True,
)
print("Working directory:", root)


## Notebook bootstrap (Kaggle / out-of-order cells)

Run this right after the repository cell if you see `ModuleNotFoundError: No module named 'ghostexec'`. It finds `pyproject.toml` + `models.py` (including under `/kaggle/input/...` when the repo is a **Dataset**), `cd`s there, adds the repo to `sys.path`, and runs `pip install -e .` via `notebook_setup.py`.


In [ ]:
# Enables `from ghostexec...` when cwd is not the repo (common on Kaggle: /kaggle/working).
import importlib.util
import sys
from pathlib import Path


def _scan_repo_roots():
    for d in (
        Path.cwd(),
        Path.cwd() / "ghostexec",
        Path("/content/ghostexec"),
        Path("/kaggle/working/ghostexec"),
        Path("/kaggle/working"),
    ):
        yield d
        yield d / "ghostexec"
    kin = Path("/kaggle/input")
    if kin.is_dir():
        for ds in sorted(kin.iterdir()):
            yield ds
            yield ds / "ghostexec"


def _find_repo_root() -> Path | None:
    for p in _scan_repo_roots():
        try:
            if (p / "pyproject.toml").is_file() and (p / "models.py").is_file():
                return p.resolve()
        except OSError:
            continue
    return None


_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "ghostexec repo not found (pyproject.toml + models.py). "
        "On Kaggle: add this repo as a Dataset under /kaggle/input, or enable Internet and run the clone cell."
    )
_setup = _root / "notebook_setup.py"
if not _setup.is_file():
    raise RuntimeError(f"Missing {_setup} — pull the latest ghostexec repo.")
_spec = importlib.util.spec_from_file_location("_ghostexec_nb_setup", _setup)
_mod = importlib.util.module_from_spec(_spec)
assert _spec and _spec.loader
sys.modules.setdefault("_ghostexec_nb_setup", _mod)
_spec.loader.exec_module(_mod)
ROOT = _mod.ensure_ghostexec_on_path()
print("ghostexec bootstrap OK:", ROOT)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchcodec"], check=False)


In [ ]:
# !pip install -q -e /content/ghostexec

## Optional SFT: briefing → JSON action

Synthetic pairs from `smart_action` in `training/train.py` so the model learns valid `GhostexecAction` JSON before GRPO.


In [ ]:
import json
import random
from pathlib import Path

from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.train import smart_action

ROOT = Path.cwd().resolve()
scenario = ROOT / "scenarios" / "phase2_core.json"
env = GhostexecEnvironment(scenario)

INSTRUCTION = (
    "You are an executive assistant agent. Reply with EXACTLY one JSON object (no markdown fences) "
    "with keys matching GhostexecAction: action_type (string), and optional email_id, message_body, "
    "meeting_id, new_time, reason, task_id, contact_name, message."
)


def build_sft_rows(n: int, seed: int = 0) -> list[dict]:
    rng = random.Random(seed)
    rows: list[dict] = []
    for _ in range(n):
        obs = env.reset()
        act = smart_action(obs, rng)
        user = INSTRUCTION + "\n\n=== BRIEFING ===\n" + (obs.echoed_message or "")
        assistant = json.dumps(act.model_dump(mode="json"), ensure_ascii=False)
        rows.append({"user": user, "assistant": assistant})
    return rows


sft_rows = build_sft_rows(min(SFT_SAMPLES, 512), seed=42)
sft_path = ROOT / "outputs" / "training" / "colab_sft_messages.jsonl"
sft_path.parent.mkdir(parents=True, exist_ok=True)
with sft_path.open("w", encoding="utf-8") as fh:
    for r in sft_rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Wrote", sft_path, "rows", len(sft_rows))


### SFT train (`SFTTrainer`)

Skipped when `RUN_SFT` is false. Uses 4-bit LoRA via Unsloth.


In [ ]:
import torch
from datasets import Dataset

model = None
tokenizer = None

if RUN_SFT:
    from unsloth import FastLanguageModel
    from trl import SFTConfig, SFTTrainer

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    def to_text(example):
        msgs = [
            {"role": "user", "content": example["user"]},
            {"role": "assistant", "content": example["assistant"]},
        ]
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

    ds = Dataset.from_list(sft_rows)
    ds = ds.map(to_text, remove_columns=["user", "assistant"])

    _cuda = torch.cuda.is_available()
    _bf16 = _cuda and torch.cuda.is_bf16_supported()
    sft_args = SFTConfig(
        output_dir=str(ROOT / "outputs" / "training" / "unsloth_sft_ckpt"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        warmup_steps=1,
        max_steps=SFT_MAX_STEPS,
        logging_steps=1,
        learning_rate=2e-4,
        fp16=_cuda and not _bf16,
        bf16=_bf16,
        dataset_text_field="text",
        report_to="none",
    )
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=sft_args)
    trainer.train()
    print("SFT done.")
else:
    print("Skipping SFT (RUN_SFT=0).")


## Optional: constrained JSON decoding (biggest sampling speedup)

Every unparsable GRPO sample is wasted compute. `patch_model_for_json_generation` wraps `model.generate` so every completion — including the ones TRL samples inside `GRPOTrainer` — is constrained to the `GhostexecAction` JSON schema.

Backends (install one):

```
pip install -q lm-format-enforcer   # preferred: works with the Unsloth tokenizer as-is
# or:
pip install -q outlines             # secondary backend
```

Set `GHOSTEXEC_CONSTRAIN_JSON=1` before this cell to activate it; the cell is otherwise a no-op.

In [ ]:
import os

if os.environ.get("GHOSTEXEC_CONSTRAIN_JSON", "0") not in ("", "0", "false", "False"):
    from training.constrained_decode import patch_model_for_json_generation

    try:
        _unpatch_json = patch_model_for_json_generation(model, tokenizer)
        print("Constrained JSON decoding enabled.")
    except ImportError as e:
        print("Constrained decoding skipped:", e)
else:
    print("Set GHOSTEXEC_CONSTRAIN_JSON=1 before this cell to enable JSON-constrained sampling.")


## Baseline (before GRPO) — artifact capture

Evaluates the **untrained** model on the held-out `vip_meltdown.json` via `training/eval_harness.py` and saves one fixed-briefing action sample. We reuse the same `llm_text_policy` at the end for the after-training pass so the comparison is apples-to-apples.

Outputs:

- `outputs/eval/llm_before.json` — aggregate metrics (mean return, format-valid, VIP-critical-reply, conflicts-resolved, per-channel).
- `outputs/eval/before_sample.txt` — the model's first action on `phase2_core.json`.

These feed the before/after table at the bottom of the notebook (the "observable evidence of training progress" artifact).


In [ ]:
from pathlib import Path

import torch

from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.eval_harness import evaluate, text_policy_as_action, write_report

ROOT = Path.cwd().resolve()
FIXED_BRIEF = ROOT / "scenarios" / "phase2_core.json"

LLM_POLICY_SYSTEM = (
    "You are Ghostexec. Respond with EXACTLY one JSON object matching GhostexecAction "
    "(no markdown fences, no prose). Keys: action_type plus any of email_id, message_body, "
    "meeting_id, new_time, reason, task_id, contact_name, message."
)


def llm_text_policy(obs, rng):
    messages = [
        {"role": "system", "content": LLM_POLICY_SYSTEM},
        {"role": "user", "content": obs.echoed_message or ""},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=192,
            do_sample=False,
            pad_token_id=getattr(tokenizer, "pad_token_id", tokenizer.eos_token_id),
        )
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


_eval_dir = ROOT / "outputs" / "eval"
_eval_dir.mkdir(parents=True, exist_ok=True)

llm_policy = text_policy_as_action(llm_text_policy)
before_report = evaluate(llm_policy, episodes_per_scenario=3, max_steps=8)
write_report(before_report, _eval_dir / "llm_before.json")
print("BEFORE (held-out LLM eval):", before_report.aggregate())

_env = GhostexecEnvironment(FIXED_BRIEF)
_obs = _env.reset()
before_sample = llm_text_policy(_obs, None)
(_eval_dir / "before_sample.txt").write_text(before_sample, encoding="utf-8")
print("\nBEFORE action on phase2_core.json:\n" + before_sample)


## GRPO with Ghostexec reward

Core reward: `ghostexec_env_step_reward` (fresh env reset → parse JSON → one `step()`). We pass it as part of a **list** of reward channels so GRPO also sees format validity, within-group diversity, ID relevance, and a VIP-critical reply bonus — all additive side channels that keep the 0.35 / 0.35 / 0.30 core blend intact. Knobs via env vars: `GHOSTEXEC_CURRICULUM` (`easy|mid|hard|all`), `GHOSTEXEC_PERTURB=1`, `GHOSTEXEC_REWARD_KSTEPS` (k-step scripted lookahead), `GHOSTEXEC_REWARD_GAMMA`.

**Optional multi-turn reward**: set `GHOSTEXEC_MULTITURN=1` (and optionally `GHOSTEXEC_MULTITURN_TURNS=3`, `GHOSTEXEC_MULTITURN_GAMMA=0.9`) to swap the single-step scalar for `make_multiturn_reward`, which uses the model itself to generate k-1 follow-up JSON actions after the sampled completion. Heavier but credits first actions that enable good follow-ups. Dataset uses the same **chat-style** `prompt` list as the OpenEnv 2048 notebook (`[{role, content}]`).


In [ ]:
import os
import torch
from datasets import Dataset
from pathlib import Path
from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.grpo_ghostexec_reward import (
    ghostexec_env_step_reward,
    reward_format_valid,
    reward_group_diversity,
    reward_id_relevance,
    reward_vip_critical_reply_bonus,
)
from training.multiturn_reward import make_multiturn_reward
from trl import GRPOConfig, GRPOTrainer


def _grpo_env_int(name: str, fallback: int) -> int:
    """Read int from env; use ``fallback`` if unset or invalid.

    Lets you set ``os.environ['GHOSTEXEC_GRPO_MAX_STEPS']`` in a later cell without
    re-running the knobs cell — values are re-read here right before GRPO builds.
    """
    try:
        v = os.environ.get(name, "").strip()
        if not v:
            return fallback
        return int(v)
    except ValueError:
        return fallback


def _grpo_env_float(name: str, fallback: float) -> float:
    """Read float from env; use ``fallback`` if unset or invalid."""
    try:
        v = os.environ.get(name, "").strip()
        if not v:
            return fallback
        return float(v)
    except ValueError:
        return fallback


GRPO_MAX_STEPS = _grpo_env_int("GHOSTEXEC_GRPO_MAX_STEPS", GRPO_MAX_STEPS)
GRPO_DATASET_ROWS = _grpo_env_int("GHOSTEXEC_GRPO_ROWS", GRPO_DATASET_ROWS)
NUM_GENERATIONS = _grpo_env_int("GHOSTEXEC_NUM_GENERATIONS", NUM_GENERATIONS)
GRPO_TEMPERATURE = _grpo_env_float("GHOSTEXEC_GRPO_TEMPERATURE", 1.1)
GRPO_MAX_COMPLETION_LENGTH = max(
    128,
    _grpo_env_int("GHOSTEXEC_GRPO_MAX_COMPLETION_LENGTH", 256),
)
GRPO_LEARNING_RATE = _grpo_env_float("GHOSTEXEC_GRPO_LR", 5e-6)
print(
    "GRPO (env re-sync):",
    f"max_steps={GRPO_MAX_STEPS}, dataset_rows={GRPO_DATASET_ROWS}, num_generations={NUM_GENERATIONS}, "
    f"temperature={GRPO_TEMPERATURE}, max_completion_cap={GRPO_MAX_COMPLETION_LENGTH}, lr={GRPO_LEARNING_RATE}",
)

ROOT = Path.cwd().resolve()
scenario = ROOT / "scenarios" / "phase2_core.json"
_env = GhostexecEnvironment(scenario)
_brief = _env.reset().echoed_message or ""
GRPO_PROMPT = INSTRUCTION + "\n\n=== BRIEFING ===\n" + _brief

user_msg = {"role": "user", "content": GRPO_PROMPT.strip()}
grpo_ds = Dataset.from_list([{"prompt": [user_msg]}] * GRPO_DATASET_ROWS)

if model is None or tokenizer is None:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

maximum_length = len(tokenizer.apply_chat_template(grpo_ds[0]["prompt"], add_generation_prompt=True))
max_prompt_length = maximum_length + 4
max_completion_length = min(max(128, MAX_SEQ_LENGTH - max_prompt_length), GRPO_MAX_COMPLETION_LENGTH)

_cuda = torch.cuda.is_available()
_bf16 = _cuda and torch.cuda.is_bf16_supported()
training_args = GRPOConfig(
    temperature=GRPO_TEMPERATURE,
    learning_rate=GRPO_LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=GRPO_MAX_STEPS,
    save_steps=max(1, GRPO_MAX_STEPS),
    report_to="none",
    output_dir=str(ROOT / "outputs" / "training" / "unsloth_grpo_ckpt"),
    fp16=_cuda and not _bf16,
    bf16=_bf16,
    remove_unused_columns=False,
)

trainer_grpo = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=(
        [
            make_multiturn_reward(
                model,
                tokenizer,
                num_turns=int(os.environ.get("GHOSTEXEC_MULTITURN_TURNS", "3")),
                gamma=float(os.environ.get("GHOSTEXEC_MULTITURN_GAMMA", "0.9")),
            ),
            reward_format_valid,
            reward_group_diversity,
            reward_id_relevance,
            reward_vip_critical_reply_bonus,
        ]
        if os.environ.get("GHOSTEXEC_MULTITURN", "0") not in ("", "0", "false", "False")
        else [
            ghostexec_env_step_reward,
            reward_format_valid,
            reward_group_diversity,
            reward_id_relevance,
            reward_vip_critical_reply_bonus,
        ]
    ),
    args=training_args,
    train_dataset=grpo_ds,
)
trainer_grpo.train()
print("GRPO finished.")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
_plots_dir = ROOT / "outputs" / "plots"
_plots_dir.mkdir(parents=True, exist_ok=True)

try:
    log_hist = trainer_grpo.state.log_history

    main_keys = [k for h in log_hist for k in h if "reward" in k.lower() and "mean" in k]
    rewards = []
    for h in log_hist:
        for k in main_keys:
            if k in h:
                rewards.append(h[k])
                break
    if rewards:
        plt.figure(figsize=(8, 3))
        plt.plot(rewards, marker="o")
        plt.xlabel("log step")
        plt.ylabel("mean reward")
        plt.title("GRPO (Ghostexec) — mean reward per log step")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        curve_path = _plots_dir / "grpo_reward_curve.png"
        plt.savefig(curve_path, dpi=140, bbox_inches="tight")
        plt.show()
        print("Saved", curve_path)
    else:
        print("No reward mean keys in log_history; sample keys:", list(log_hist[0].keys()) if log_hist else [])

    channel_candidates = set()
    for h in log_hist:
        for k in h:
            kl = k.lower()
            if ("reward" in kl) and any(tag in kl for tag in ("func", "/", "format", "diversity", "id_relevance", "vip_critical", "env_step", "multiturn")):
                channel_candidates.add(k)
    channel_keys = sorted(channel_candidates)
    if channel_keys:
        plt.figure(figsize=(9, 4))
        for k in channel_keys:
            vals = [h.get(k) for h in log_hist if k in h and h.get(k) is not None]
            if vals:
                plt.plot(vals, label=k.split("/")[-1], marker=".", alpha=0.8)
        plt.xlabel("log step")
        plt.ylabel("reward")
        plt.title("GRPO reward channels (anti-hack evidence)")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=8, loc="best")
        plt.tight_layout()
        channels_path = _plots_dir / "grpo_reward_channels.png"
        plt.savefig(channels_path, dpi=140, bbox_inches="tight")
        plt.show()
        print("Saved", channels_path)
    else:
        print("No per-channel reward keys in log_history yet.")
except Exception as e:
    print("Plot skip:", e)


## Held-out evaluation — AFTER training + before/after comparison

Runs the trained policy on `EVAL_SCENARIOS` (`vip_meltdown.json`), then diffs it against the BEFORE eval we captured earlier. Also keeps the scripted `smart_action` baseline as a reference.

Writes the artifacts that satisfy the **"observable evidence of training progress"** rubric:

- `outputs/eval/llm_after.json` — same schema as `llm_before.json`.
- `outputs/eval/scripted_baseline.json` — reference baseline from the hand-written policy.
- `outputs/eval/after_sample.txt` — trained model's action on the same `phase2_core.json` briefing used for the BEFORE sample.
- `outputs/eval/before_after.csv` — 7-row comparison table (the thing reviewers look at).


In [ ]:
import csv
from pathlib import Path

from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.eval_harness import evaluate, scripted_smart_policy, text_policy_as_action, write_report

ROOT = Path.cwd().resolve()
_eval_dir = ROOT / "outputs" / "eval"
_eval_dir.mkdir(parents=True, exist_ok=True)

scripted = evaluate(scripted_smart_policy(), episodes_per_scenario=2, max_steps=8)
write_report(scripted, _eval_dir / "scripted_baseline.json")
print("Scripted baseline:", scripted.aggregate())

after_report = evaluate(text_policy_as_action(llm_text_policy), episodes_per_scenario=3, max_steps=8)
write_report(after_report, _eval_dir / "llm_after.json")
print("\nAFTER (held-out LLM eval):", after_report.aggregate())

_env = GhostexecEnvironment(FIXED_BRIEF)
_obs = _env.reset()
after_sample = llm_text_policy(_obs, None)
(_eval_dir / "after_sample.txt").write_text(after_sample, encoding="utf-8")

METRIC_KEYS = (
    "mean_return",
    "format_valid_rate",
    "vip_critical_first_reply_rate",
    "conflicts_resolved_rate",
    "mean_channel_conflict",
    "mean_channel_relationship",
    "mean_channel_task",
)
b = before_report.aggregate()
a = after_report.aggregate()
rows = [("metric", "before", "after", "delta")]
for k in METRIC_KEYS:
    rows.append((k, round(b.get(k, 0.0), 3), round(a.get(k, 0.0), 3), round(a.get(k, 0.0) - b.get(k, 0.0), 3)))

csv_path = _eval_dir / "before_after.csv"
with csv_path.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    for row in rows:
        writer.writerow(row)
print("\nSaved", csv_path)

widths = [max(len(str(r[i])) for r in rows) for i in range(4)]
for r in rows:
    print("  " + "  ".join(str(c).ljust(widths[i]) for i, c in enumerate(r)))

print("\nBEFORE action on phase2_core.json:\n" + before_sample)
print("\nAFTER action on phase2_core.json:\n" + after_sample)

try:
    import pandas as pd

    df = pd.DataFrame(rows[1:], columns=rows[0])
    display(df)
except Exception:
    pass
